# Clase 1 — Anatomía de un prompt

Un **prompt** es la instrucción que le das a un modelo de lenguaje. La diferencia entre una respuesta útil y una vaga casi siempre está en cómo escribiste esa instrucción — no en el modelo.

En esta clase vamos a descomponer un prompt en sus partes, entender por qué cada una importa, y comparar en tiempo real qué pasa cuando las usás bien o mal.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Las cinco partes de un prompt |
| 3 | Armar el wrapper LLM |
| 4 | Experimento: prompt vacío vs. prompt estructurado |
| 5 | Ajustar cada parte por separado |
| 6 | Actividad práctica |

```
DIAGRAMA ASCII — Wrapper `llamar_llm()` del notebook
====================================================

                 +----------------------------------+
                 | Usuario / notebook               |
                 | prompt, system_prompt, temp,     |
                 | max_tokens                       |
                 +----------------+-----------------+
                                  |
                                  v
                    +-----------------------------+
                    | llamar_llm(...)             |
                    |-----------------------------|
                    | if BACKEND == "gemini"      |
                    | elif BACKEND == "local"     |
                    +-------------+---------------+
                                  |
              +-------------------+-------------------+
              |                                       |
              v                                       v
+-------------------------------+      +----------------------------------+
| Rama Gemini                   |      | Rama local                       |
|-------------------------------|      |----------------------------------|
| _cliente_gemini               |      | _llm_local                       |
| .models.generate_content(...) |      | .create_chat_completion(...)     |
|                               |      |                                  |
| model = GEMINI_MODEL          |      | messages = [system, user]        |
| contents = prompt             |      | temperature                      |
| system_instruction            |      | max_tokens                       |
| temperature                   |      +----------------+-----------------+
| max_output_tokens             |                       |
+---------------+---------------+                       |
                |                                       |
                +-------------------+-------------------+
                                    |
                                    v
                     +-------------------------------+
                     | Respuesta del modelo          |
                     | .text.strip()  /  content     |
                     +---------------+---------------+
                                     |
                                     v
                     +-------------------------------+
                     | Devuelve un string final      |
                     | listo para imprimir           |
                     +-------------------------------+


Idea clave
----------
El notebook usa una sola función (`llamar_llm`) como interfaz común.
Así, el resto de las celdas no necesita cambiar aunque cambie el backend.
```

---
## 1. Configuración del entorno

Este notebook puede correr de dos formas:

| Backend | Qué necesitás | Cuándo usarlo |
|---|---|---|
| `local` | llama.cpp instalado + modelo GGUF descargado | Sin internet, máquina con ≥4 GB RAM |
| `gemini` | Cuenta Google, API key gratuita | Recomendado si es tu primera vez |

### Obtener tu API key de Gemini (solo una vez)

1. Entrá a [aistudio.google.com](https://aistudio.google.com) con tu cuenta Google.
2. Hacé clic en **Get API key** → **Create API key**.
3. Copiá la clave (empieza con `AIza...`).

### Guardar la key de forma segura

**Nunca** escribas la clave directamente en el código: si subís el notebook a GitHub, queda expuesta.
Lo correcto es guardarla en un archivo `.env` en la carpeta del proyecto:

```bash
# Ejecutá esto UNA VEZ en tu terminal, desde la carpeta del proyecto
echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
```

El archivo `.env` queda en tu máquina y **no** se sube al repositorio.
Si no querés crear el archivo, la celda siguiente te pide la clave de forma interactiva.

In [ ]:
# ─── Instalación de dependencias (solo la primera vez) ────────────────────────
# Descomentá la línea que necesites:
!pip install google-genai python-dotenv          # para backend Gemini
!pip install llama-cpp-python huggingface-hub    # para backend local
print("Listo para configurar.")

In [1]:
# ─── Elegí tu backend ─────────────────────────────────────────────────────────
BACKEND = "local"   # Cambiá a "local" si usás llama.cpp

In [2]:
import os
import getpass

# Modelos Gemini disponibles con tier gratuito:
GEMINI_MODEL = "gemini-2.0-flash-lite"   # rápido y gratuito
# GEMINI_MODEL = "gemma-4-26b-a4b-it"   # alternativa de Google DeepMind

# ─── Cargar API key ───────────────────────────────────────────────────────────
if BACKEND == "gemini":
    # Primero intenta leer del archivo .env
    try:
        from dotenv import load_dotenv
        load_dotenv()   # carga variables desde el archivo .env si existe
    except ImportError:
        pass  # si no está python-dotenv, sigue igual

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

    # Si no encontró la key en .env, la pide de forma interactiva
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini (no se muestra): ")

print(f"Backend seleccionado: {BACKEND}")
print("API key: OK" if BACKEND == "gemini" and GEMINI_API_KEY else "(modo local)")

Backend seleccionado: local
(modo local)


---
## 2. Las cinco partes de un prompt

Un prompt bien armado puede tener hasta cinco componentes. No todos son obligatorios en todas las situaciones, pero conocerlos te permite diagnosticar rápido por qué una respuesta salió mal.

| Componente | Qué hace | Ejemplo |
|---|---|---|
| **Rol** | Le dice al modelo *quién* debe ser | `"Sos un contador público con 10 años de experiencia"` |
| **Contexto** | Información de fondo relevante para la tarea | `"El cliente tiene una PyME de importación"` |
| **Instrucción** | La tarea concreta a realizar | `"Explicá las deducciones posibles en el impuesto a las ganancias"` |
| **Formato** | Cómo debe estructurarse la respuesta | `"En 3 puntos, sin tecnicismos"` |
| **Output esperado** | Qué debe incluir o evitar la respuesta | `"Incluí un ejemplo numérico simple al final"` |

_
> 💡 El componente que más se omite (y más falta hace) es el **formato**. Sin él, el modelo elige cómo estructurar la respuesta — y no siempre elige bien para tu caso de uso.

---
## 3. Armar el wrapper LLM

Para no repetir código en cada experimento, armamos una función única `llamar_llm()` que funciona con cualquiera de los dos backends. La firma es siempre la misma: preguntás y recibís texto.

In [3]:
# ─── Inicializar el cliente según el backend elegido ─────────────────────────

if BACKEND == "gemini":
    from google import genai
    from google.genai import types

    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
    print("Cliente Gemini inicializado.")

elif BACKEND == "local":
    import time
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    # Granite-4.0-350M — modelo instruct con chat template
    REPO_ID = "ibm-granite/granite-4.0-350m-GGUF"
    FILENAME = "granite-4.0-350m-Q4_K_M.gguf"

    print("Descargando modelo local (puede tardar la primera vez)...")
    ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)
    print("Modelo local listo.")

else:
    raise ValueError(f"Backend '{BACKEND}' no reconocido. Usá 'gemini' o 'local'.")

c:\Users\maria\GitHub\clases\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando modelo local (puede tardar la primera vez)...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Modelo local listo.


In [6]:
def llamar_llm(
    prompt,
    system_prompt="follow the instructions. answer in spanish.",
    temperature=0.7,
    max_tokens=200
):
    """Envía un prompt al modelo configurado y devuelve la respuesta como string."""

    if BACKEND == "gemini":
        respuesta = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return respuesta.text.strip()

    elif BACKEND == "local":
        # Granite-4.0-350M es un modelo instruct con chat template.
        # create_chat_completion() aplica el template automáticamente.
        respuesta = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return respuesta["choices"][0]["message"]["content"].strip()


# Prueba rápida de conexión
print(llamar_llm("Decí 'Hola, estoy funcionando'", max_tokens=20))

¡Hola, estas funcionando!


---
## 4. Experimento: prompt vacío vs. prompt estructurado

Vamos a hacer la misma consulta de tres formas distintas. La tarea en los tres casos es la misma: que el modelo explique qué es una red neuronal. Solo cambia cómo se la pedimos.

In [7]:
# ─── Versión 1: prompt mínimo — sin estructura ────────────────────────────────
prompt_minimo = "Explicá redes neuronales."

print("=" * 60)
print("VERSIÓN 1 — Prompt mínimo")
print("=" * 60)
print(llamar_llm(prompt_minimo))
print()

VERSIÓN 1 — Prompt mínimo
Las redes neuronales son una serie de neuronas artificiales diseñadas para realizar una tarea específica y procesar información. Estas neuronas se unen juntas para crear una red neural compleja que puede reconocer y procesar información de manera efectiva. La función principal de las redes neuronales es procesar y analizar datos y, a menudo, se utilizan para tareas como reconocimiento de imágenes, procesamiento de lenguaje natural, clasificación y otras.



In [8]:
# ─── Versión 2: con rol y formato, sin contexto ───────────────────────────────
prompt_con_rol = """Sos un profesor de tecnología.
Explicá qué es una red neuronal en exactamente 3 puntos breves."""

print("=" * 60)
print("VERSIÓN 2 — Con rol y formato")
print("=" * 60)
print(llamar_llm(prompt_con_rol))
print()

VERSIÓN 2 — Con rol y formato
Una red neuronal es un modelo de aprendizaje de máquina que utiliza algoritmos para aprender características de información y realizar tareas de inteligencia artificial.



In [9]:
# ─── Versión 3: prompt completo con todos los componentes ─────────────────────
prompt_completo = """Rol: Sos un profesor universitario de inteligencia artificial.
Contexto: Estás explicando a estudiantes adultos que trabajan en empresas y no tienen
  experiencia previa en programación.
Instrucción: Explicá qué es una red neuronal artificial.
Formato: 3 bullets, máximo 2 líneas cada uno, sin fórmulas matemáticas.
Output esperado: Terminá con una analogía cotidiana que resuma la idea principal."""

print("=" * 60)
print("VERSIÓN 3 — Prompt completo")
print("=" * 60)
print(llamar_llm(prompt_completo))
print()

VERSIÓN 3 — Prompt completo
Una red neuronal artificial es una red neural que se utiliza para modelar y aprender las relaciones entre los datos. Para los estudiantes, puede pensar de un algoritmo que procesa varios datos y realiza un análisis.



> 💡 **Para discutir:** ¿En qué versión la respuesta fue más útil para el público descrito? ¿Qué componente del prompt completo hizo más diferencia? ¿Hubo algún componente que parecía redundante?

---
## 5. Ajustar cada parte por separado

Ahora vamos a ver cómo cambia la respuesta cuando modificamos *un solo componente* a la vez. Esto entrena la intuición para saber dónde tocar cuando una respuesta no es la que esperabas.

In [10]:
# ─── Efecto del ROL ───────────────────────────────────────────────────────────
# La instrucción es idéntica, solo cambia quién se supone que responde.

instruccion_base = "Explicá en 2 oraciones por qué es importante proteger los datos personales."

roles = [
    "Sos un abogado especialista en privacidad.",
    "Sos un técnico de sistemas en una empresa mediana.",
    "Sos un periodista que escribe para el público general."
]

for rol in roles:
    print(f"Rol: {rol}")
    print("-" * 50)
    print(llamar_llm(instruccion_base, system_prompt=rol, max_tokens=120))
    print()

Rol: Sos un abogado especialista en privacidad.
--------------------------------------------------
1. Para mantener la confidencialidad: Proteger los datos personales es importante para garantizar la privacidad y seguridad de los datos personales sensibles que contienen. La protección de la privacidad ayuda a mantener la confidencialidad, reduciendo el riesgo de ciberataque y mal uso, garantizando la integridad de los datos personales.

2. Para respetar la autonomía y la dignidad: Proteger los datos personales es importante para respetar la autonomía y la dignidad de los individuos. Proteger

Rol: Sos un técnico de sistemas en una empresa mediana.
--------------------------------------------------
Un técnico de sistemas en una empresa mediana es importante proteger los datos personales porque las pérdidas de información sensacionalistas pueden tener graves consecuencias para la reputación de la empresa y la rentabilidad. Proteger los datos personales es importante para mantener la conf

In [11]:
# ─── Efecto del FORMATO ───────────────────────────────────────────────────────
# El contenido pedido es el mismo, pero el formato cambia completamente la usabilidad.

contenido = "Enumerá las ventajas de usar modelos de lenguaje en atención al cliente."

formatos = [
    "Respondé en prosa, como si fuera un párrafo de informe ejecutivo.",
    "Respondé en una lista de 4 ítems concisos, sin introducción.",
    "Respondé como una tabla con dos columnas: Ventaja | Por qué importa."
]

for fmt in formatos:
    print(f"Formato: {fmt}")
    print("-" * 50)
    print(llamar_llm(f"{contenido}\n{fmt}", max_tokens=180))
    print()

Formato: Respondé en prosa, como si fuera un párrafo de informe ejecutivo.
--------------------------------------------------
Los modelos de lenguaje en atención al cliente ofrecen una serie de ventajas, como mejorar la eficiencia de las conversaciones y aumentar la confianza del cliente. Ofrecen la posibilidad de generar contenido de texto y voz, lo que permite a los hablantes adaptarse a diversos contextos y tamaños de discursos. Además, estos modelos pueden analizar el sentimiento y la intención del cliente, proporcionando información valiosa para tomar decisiones informadas. También facilitan la automatización del procesamiento de los datos, lo que permite a los modelos de lenguaje procesar grandes cantidades de datos y analizar grandes volúmenes de información. Finalmente, estos modelos de lenguaje pueden generar contenido como resumen, recomendaciones o apoyo al cliente, lo que facilita la mejora continua del servicio al cliente

Formato: Respondé en una lista de 4 ítems concisos

---
## 6. Actividad práctica

Tomá la siguiente tarea y construí un prompt completo usando los cinco componentes. Después compará tu resultado con el prompt mínimo que se incluye abajo.

**Tarea:** pedirle al modelo que explique cómo funciona una tarjeta de crédito para alguien que nunca tuvo una usando el modelo local.

In [12]:
# ─── Prompt mínimo (punto de partida) ────────────────────────────────────────
prompt_minimo_actividad = "Explicá cómo funciona una tarjeta de crédito."

print("RESULTADO CON PROMPT MÍNIMO:")
print("-" * 50)
print(llamar_llm(prompt_minimo_actividad, max_tokens=150))
print()

RESULTADO CON PROMPT MÍNIMO:
--------------------------------------------------
Una tarjeta de crédito es un tipo de cuenta bancaria que permite a los clientes pagar los préstamos con un pago mensual o trimestral. Para obtener la tarjeta, los clientes deben presentar documentos personales que incluyen una copia de identificación, teléfono de dirección y información de contacto. Una vez que se les solicite la tarjeta, los clientes reciben un documento con la información de la tarjeta, que les permite acceder a su cuenta de crédito y realizar transacciones con ellos. La tarjeta de crédito es un servicio que nos da acceso a la información financiera y a los préstamos, y nos permite pagar los préstamos con una tarjeta digital que nos permite



In [15]:
def llamar_llm(
    prompt,
    system_prompt="Responde en español, simple y conciso.",
    temperature=0.7,
    max_tokens=200 ):


    respuesta = _llm_local.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return respuesta["choices"][0]["message"]["content"].strip()


# Prueba rápida de conexión
print(llamar_llm("Decí 'Hola, estoy funcionando' y nada más.", max_tokens=20))

Saludo! Estoy aquí y estoy feliz de ayudarte con tus necesidades.


In [ ]:
# TODO: Completá el prompt con los 5 componentes
# Reemplazá cada "..." con tu texto

"""Rol: Sos un contador que explica de forma sencilla a una persona que nunca tuvo una tarjeta de crédito.
Contexto: ...
Instrucción: ...
Formato: ...
Output esperado: ...
"""

print("RESULTADO CON TU PROMPT:")
print("-" * 50)
print(llamar_llm(mi_prompt, max_tokens=300))